In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from joblib import load
from scipy.stats import pearsonr

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# ─── LOAD DATA FOR BOTH DATASETS ────────────────────────────────────────────────
df_addhealth = load('symmetry_stability_results_AddHealth.joblib')
df_banerjee = load('symmetry_stability_results_Banerjee.joblib')

# ────────────────────────────────────────────────────────────────────────────────
# Prepare data for AddHealth
df_original_add = df_addhealth[df_addhealth['removal_portion'] == 0.0][['name', 'T', 'symmetry']].rename(
    columns={'symmetry': 'symmetry_original'})
df_merged_add = df_addhealth.merge(df_original_add, on=['name', 'T'], how='left')
df_merged_add = df_merged_add[df_merged_add['removal_portion'].isin([0.1, 0.2, 0.3])]
df_merged_add = df_merged_add.dropna(subset=['symmetry_original', 'symmetry'])

# Prepare data for Banerjee
df_original_ban = df_banerjee[df_banerjee['removal_portion'] == 0.0][['name', 'T', 'symmetry']].rename(
    columns={'symmetry': 'symmetry_original'})
df_merged_ban = df_banerjee.merge(df_original_ban, on=['name', 'T'], how='left')
df_merged_ban = df_merged_ban[df_merged_ban['removal_portion'].isin([0.1, 0.2, 0.3])]
df_merged_ban = df_merged_ban.dropna(subset=['symmetry_original', 'symmetry'])

# Compute min/max for AddHealth
min_val_add = min(df_merged_add['symmetry_original'].min(), df_merged_add['symmetry'].min())
max_val_add = max(df_merged_add['symmetry_original'].max(), df_merged_add['symmetry'].max())

# Compute min/max for Banerjee
min_val_ban = min(df_merged_ban['symmetry_original'].min(), df_merged_ban['symmetry'].min())
max_val_ban = max(df_merged_ban['symmetry_original'].max(), df_merged_ban['symmetry'].max())

# ────────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 12), sharex='row', sharey='row')
portions = [0.1, 0.2, 0.3]

# Row 0: AddHealth
for i, portion in enumerate(portions):
    ax = axes[0, i]
    subset = df_merged_add[df_merged_add['removal_portion'] == portion]

    sns.kdeplot(
        data=subset, x="symmetry_original", y="symmetry",
        fill=True, cmap="Blues", thresh=0.05, levels=15, bw_adjust=0.8, ax=ax
    )

    ax.plot([min_val_add, max_val_add], [min_val_add, max_val_add],
            'k--', linewidth=0.8, alpha=0.6)

    corr, _ = pearsonr(subset['symmetry_original'], subset['symmetry'])
    textbox = dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.85)
    ax.text(0.05, 0.95, f'Pearson $r = {corr:.2f}$',
            transform=ax.transAxes, fontsize=12, verticalalignment='top', bbox=textbox)

    ax.set_title(f'AddHealth, Removal = {portion*100:.0f}%', fontsize=14)
    ax.set_xlabel(r"$\Xi_{\mathrm{original}}$", fontsize=16)
    ax.set_ylabel(r"$\Xi_{\mathrm{after\ removal}}$", fontsize=16)
    ax.set_xlim(min_val_add - 0.05, max_val_add + 0.05)
    ax.set_ylim(min_val_add - 0.05, max_val_add + 0.05)
    ax.set_aspect('equal')

# Row 1: Banerjee
for i, portion in enumerate(portions):
    ax = axes[1, i]
    subset = df_merged_ban[df_merged_ban['removal_portion'] == portion]

    sns.kdeplot(
        data=subset, x="symmetry_original", y="symmetry",
        fill=True, cmap="Blues", thresh=0.05, levels=15, bw_adjust=0.8, ax=ax
    )

    ax.plot([min_val_ban, max_val_ban], [min_val_ban, max_val_ban],
            'k--', linewidth=0.8, alpha=0.6)

    corr, _ = pearsonr(subset['symmetry_original'], subset['symmetry'])
    textbox = dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.85)
    ax.text(0.05, 0.95, f'Pearson $r = {corr:.2f}$',
            transform=ax.transAxes, fontsize=12, verticalalignment='top', bbox=textbox)

    ax.set_title(f'Banerjee, Removal = {portion*100:.0f}%', fontsize=14)
    ax.set_xlabel(r"$\Xi_{\mathrm{original}}$", fontsize=16)
    ax.set_ylabel(r"$\Xi_{\mathrm{after\ removal}}$", fontsize=16)
    ax.set_xlim(min_val_ban - 0.05, max_val_ban + 0.05)
    ax.set_ylim(min_val_ban - 0.05, max_val_ban + 0.05)
    ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('kde_symmetry_original_vs_removed_subfigures_combined.png', dpi=300, bbox_inches='tight')
plt.show()